# Auditoria y limpieza de datos GMN

Este cuaderno registra la carga, inspección y limpieza de los datos utilizados en el proyeco de las Gemínidas.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Definir la raíz del proyecto y las rutas de las carpetas que se utilizarán

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
TABLES_DIT = PROJECT_ROOT / "reports" / "tables"

PROJECT_ROOT

PosixPath('/Users/alexn/Documents/Uni_Itesm/Tercer semestre/F1009/Investigacion')

In [9]:
# Construir la ruta del archivo de 2019 y comprobar que fue descargado correctamente

raw_file = RAW_DIR / "traj_summary_monthly_201912.txt"

print("Archivo encontrado: ", raw_file.exists())
print("Ruta: ", raw_file)
print("Tamaño en MB: ", raw_file.stat().st_size / 1_000_000)

Archivo encontrado:  True
Ruta:  /Users/alexn/Documents/Uni_Itesm/Tercer semestre/F1009/Investigacion/data/raw/traj_summary_monthly_201912.txt
Tamaño en MB:  10.328228


In [21]:
# Mostrar las primeras diez líneas sin modificar ni cargar todavía toda la base

with raw_file.open(
    "r",
    encoding = "utf-8",
    errors = "replace"
) as file:
    for _ in range(10):
        print(file.readline().rstrip())

# Summary generated on 2026-10-30 23:36:11.534390+00:00 UTC

#  Unique trajectory;      Beginning      ;        Beginning          ;   IAU;   IAU;   Sol lon ;   App LST ;   RAgeo  ;   +/-  ;   DECgeo ;   +/-  ;  LAMgeo  ;   +/-  ;   BETgeo ;   +/-  ;    Vgeo  ;    +/- ;  LAMhel  ;   +/-  ;   BEThel ;   +/-  ;    Vhel  ;    +/- ;       a    ;   +/-  ;      e    ;   +/-  ;      i    ;   +/-  ;    peri   ;    +/-  ;    node   ;    +/-  ;     Pi    ;   +/-  ;      b    ;   +/-  ;      q    ;   +/-  ;      f    ;   +/-  ;      M    ;   +/-  ;       Q    ;   +/-  ;      n    ;   +/-  ;      T    ;   +/-  ; TisserandJ;   +/-  ;   RAapp  ;   +/-  ;   DECapp ;   +/-  ;  Azim +E ;   +/-  ;    Elev  ;   +/-  ;   Vinit  ;    +/- ;    Vavg  ;    +/- ;    LatBeg   ;   +/-  ;    LonBeg   ;   +/-  ;   HtBeg ;   +/-  ;    LatEnd   ;   +/-  ;    LonEnd   ;   +/-  ;   HtEnd ;   +/-  ; Duration;  Peak ;  Peak Ht;   F  ;  Mass kg;   Qc ; MedianFitErr; Beg in; End in;  Num;      Participating

#      identi

In [28]:
# Cargar el archivo GMN como DataFrame conservando sus dos filas de encabezado

df_raw = pd.read_csv(
    raw_file,
    engine = "python",
    sep = ";",
    skipinitialspace = True,
    skiprows = [0, 5, 6],
    header = [0, 1],
    na_values = ["nan", "None"],
)

print("Número de filas: ", df_raw.shape[0])
print("Número de columnas: ", df_raw.shape[1])

display(df_raw.head(3))

Número de filas:  11143
Número de columnas:  86


#  Unique trajectory  Beginning               Beginning           IAU       \
   #      identifier    Julian date               UTC Time            No code   
0  20191201000031_ghvfm     2.458819e+06  2019-12-01 00:00:31.996809  -1  ...   
1  20191201000058_C6rji     2.458819e+06  2019-12-01 00:00:58.131851  -1  ...   
2  20191201000253_e8sNU     2.458819e+06  2019-12-01 00:02:53.756370  16  HYD   

     Sol lon    App LST     RAgeo     +/-     DECgeo   ...  Peak    Peak Ht  \
       deg        deg        deg     sigma     deg     ... AbsMag      km     
0  248.210867  84.591665  143.02114  0.8854  26.41041  ...   1.19  107.9686   
1  248.211174  84.292021   68.29109  1.2096  24.78505  ...   1.03   85.2442   
2  248.212530  84.323282  118.55018  0.1622   3.93950  ...  -1.80   93.8287   

     F     Mass kg    Qc  MedianFitErr Beg in End in  Num  \
   param  tau=0.7%   deg     arcsec      FOV    FOV  stat   
0  0.500  0.000007  11.41        42.15   True   True    2   
1  0.740  0.000088   7.88        44.21   True   True    2   
2  0.641  0.000290  40.65        73.98   True   True    4   

             Participating      
                stations        
0                HR0007,HR000N  
1                HR0007,HR000N  
2  HR000F,HR000K,HR000P,HR000U  

[3 rows x 86 columns]

In [29]:
# Mostrar los primeros encabezados

print("Tipos del encabezado: ", type(df_raw.columns))
print("Número de niveles", df_raw.columns.nlevels)

display(
    pd.DataFrame(
        df_raw.columns[:15].tolist(),
        columns = ["Nivel 1: nombre", "Nivel 2: unidad"],
    )
)

Tipos del encabezado:  <class 'pandas.MultiIndex'>
Número de niveles 2


,Nivel 1: nombre,Nivel 2: unidad
0,# Unique trajectory,# identifier
1,Beginning,Julian date
2,Beginning,UTC Time
3,IAU,No
4,IAU,code
5,Sol lon,deg
6,App LST,deg
7,RAgeo,deg
8,+/-,sigma
9,DECgeo,deg


In [30]:
# Unir los dos niveles del encabezado en un solo nombre por columna

def clean_header_text(text):
    cleaned_text = str(text).replace("#", "")
    cleaned_text = " " .join(cleaned_text.split())

    return cleaned_text


flattened_columns = []

for main_name, secondary_name in df_raw.columns:
    main_name = clean_header_text(main_name)
    secondary_name = clean_header_text(secondary_name)

    if secondary_name.startswith("Unnamed"):
        final_name = main_name
    else:
        final_name = f"{main_name} ({secondary_name})"

    flattened_columns.append(final_name)


df_raw.columns = flattened_columns

for column_name in df_raw.columns[:15]:
    print(column_name)

Unique trajectory (identifier)
Beginning (Julian date)
Beginning (UTC Time)
IAU (No)
IAU (code)
Sol lon (deg)
App LST (deg)
RAgeo (deg)
+/- (sigma)
DECgeo (deg)
+/- (sigma .1)
LAMgeo (deg)
+/- (sigma .2)
BETgeo (deg)
+/- (sigma .3)


In [31]:
# Renombrar cada columna de incertidumbre usando el nombre de la variable anterior

renamed_columns = list(df_raw.columns)

for column_position, column_name in enumerate(renamed_columns):
    if column_name.startswith("+/-"):
        previous_column = renamed_columns[column_position - 1]
        parameter_name = previous_column.split(" ")[0]
        
        renamed_columns[column_position] = (
            f"{parameter_name}_sigma"
        )
        
df_raw.columns = renamed_columns

print("¿Todos los nombres son únicos?: ", df_raw.columns.is_unique)

for column_name in df_raw.columns:
    if column_name.endswith("_sigma"):
        print(column_name)

¿Todos los nombres son únicos?:  True
RAgeo_sigma
DECgeo_sigma
LAMgeo_sigma
BETgeo_sigma
Vgeo_sigma
LAMhel_sigma
BEThel_sigma
Vhel_sigma
a_sigma
e_sigma
i_sigma
peri_sigma
node_sigma
Pi_sigma
b_sigma
q_sigma
f_sigma
M_sigma
Q_sigma
n_sigma
T_sigma
TisserandJ_sigma
RAapp_sigma
DECapp_sigma
Azim_sigma
Elev_sigma
Vinit_sigma
Vavg_sigma
LatBeg_sigma
LonBeg_sigma
HtBeg_sigma
LatEnd_sigma
LonEnd_sigma
HtEnd_sigma


In [36]:
# Comprobar dimensiones, tipos generales y primeras columnas del DataFrame cargado

print("Filas: ", df_raw.shape[0])
print("Columnas: ", df_raw.shape[1])
print("Nombres únicos: ", df_raw.columns.is_unique)

print("\n Información general: ")
df_raw.info(verbose = False)

print("\n Primeras tres filas y primeras doce columnas: ")

display(df_raw.iloc[:3, :12])

Filas:  11143
Columnas:  86
Nombres únicos:  True

 Información general: 
<class 'pandas.DataFrame'>
RangeIndex: 11143 entries, 0 to 11142
Columns: 86 entries, Unique trajectory (identifier) to Participating (stations)
dtypes: bool(2), float64(78), int64(2), str(4)
memory usage: 7.2 MB

 Primeras tres filas y primeras doce columnas: 


,Unique trajectory (identifier),Beginning (Julian date),Beginning (UTC Time),IAU (No),IAU (code),Sol lon (deg),App LST (deg),RAgeo (deg),RAgeo_sigma,DECgeo (deg),DECgeo_sigma,LAMgeo (deg)
0,20191201000031_ghvfm,2.458819e+06,2019-12-01 00:00:31.996809,-1,...,248.210867,84.591665,143.02114,0.8854,26.41041,0.1940,136.82863
1,20191201000058_C6rji,2.458819e+06,2019-12-01 00:00:58.131851,-1,...,248.211174,84.292021,68.29109,1.2096,24.78505,0.4855,70.35293
2,20191201000253_e8sNU,2.458819e+06,2019-12-01 00:02:53.756370,16,HYD,248.212530,84.323282,118.55018,0.1622,3.93950,0.2990,119.83537


In [39]:
# Crear una copia del trabajo y registrar el archivo y año de procedencia

df = df_raw.copy()

df["source_file"] = raw_file.name
df["source_year"] = 2019

print("Dimensiones del original: ", df_raw.shape)
print("DImensiones de la copia: ", df.shape)

display(
    df[
        [
            "Unique trajectory (identifier)",
            "source_file",
            "source_year",
        ]
    ].head(3)
)

Dimensiones del original:  (11143, 86)
DImensiones de la copia:  (11143, 88)


,Unique trajectory (identifier),source_file,source_year
0,20191201000031_ghvfm,traj_summary_monthly_201912.txt,2019
1,20191201000058_C6rji,traj_summary_monthly_201912.txt,2019
2,20191201000253_e8sNU,traj_summary_monthly_201912.txt,2019


In [40]:
# Normalizar los textos y registrar si había campos vacíos no reconocidos como faltantes

text_columns = [
    "Unique trajectory (identifier)",
    "IAU (code)",
    "Participating (stations)",
]

missing_before = df_raw[text_columns].isna().sum()

for column in text_columns:
    df[column] = (df_raw[column].astype("string").str.strip().replace("", pd.NA))

text_report = pd.DataFrame(
    {
        "faltantes_antes": missing_before,
        "faltantes_después": df[text_columns].isna().sum(),
    }
)

display(text_report)
display(df[text_columns].head(3))

,faltantes_antes,faltantes_después
Unique trajectory (identifier),0,0
IAU (code),0,0
Participating (stations),0,0


,Unique trajectory (identifier),IAU (code),Participating (stations)
0,20191201000031_ghvfm,...,"HR0007,HR000N"
1,20191201000058_C6rji,...,"HR0007,HR000N"
2,20191201000253_e8sNU,HYD,"HR000F,HR000K,HR000P,HR000U"


In [44]:
# Convertir las fechas a datetime en UTx y comprobar su rango y valores faltantes

date_column = "Beginning (UTC Time)"

date_text = (df_raw[date_column].astype("string").str.strip().replace("", pd.NA))

df[date_column] = pd.to_datetime(date_text, format = "%Y-%m-%d %H:%M:%S.%f", errors = "raise", utc = True)

print("Tipos después de convertir: ", df[date_column].dtype)
print("Fechas faltantes antes: ", date_text.isna().sum())
print("Fechas faltantes después: ", df[date_column].isna().sum())
print("Fecha más temprana: ", df[date_column].min())
print("Fecha más tardía: ", df[date_column].max())

display(
    pd.DataFrame(
        {
            "texto_original ": df_raw[date_column].head(3),
            "fecha_interpretada ": df[date_column].head(3),
        }
    )
)

Tipos después de convertir:  datetime64[us, UTC]
Fechas faltantes antes:  0
Fechas faltantes después:  0
Fecha más temprana:  2019-12-01 00:00:31.996809+00:00
Fecha más tardía:  2019-12-31 23:57:18.061349+00:00


,texto_original,fecha_interpretada
0,2019-12-01 00:00:31.996809,2019-12-01 00:00:31.996809+00:00
1,2019-12-01 00:00:58.131851,2019-12-01 00:00:58.131851+00:00
2,2019-12-01 00:02:53.756370,2019-12-01 00:02:53.756370+00:00


In [ ]:
# Revisar los tipos principales y contar los valores de las columnas booleanas

columns_to_check = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (No)",
    "IAU (code)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
    "Participating (stations)",
]

type_report = df[columns_to_check].dtypes.to_frame(name = "tipo")

display(type_report)

boolean_columns = [
    "Beg in (FOV)",
    "End in (FOV)",
]

for column in boolean_columns:
    print("\n Columna:", column)
    display(df[column].value_counts(dropna = False))
    
print("\n Filas originales: ", len(df_raw))
print("Filas actuales: ", len(df))

,tipo
Unique trajectory (identifier),string
Beginning (UTC Time),"datetime64[us, UTC]"
IAU (No),int64
IAU (code),string
Num (stat),int64
Beg in (FOV),bool
End in (FOV),bool
Participating (stations),string



 Columna: Beg in (FOV)


Beg in (FOV)
True     11108
False       35
Name: count, dtype: int64


 Columna: End in (FOV)


End in (FOV)
True     11084
False       59
Name: count, dtype: int64


 Filas originales:  11143
Filas actuales:  11143
(11143, 88)


In [47]:
# Resumir la cantidad y el porcentaje de variables faltantes por columna

missing_report = pd.DataFrame(
    {
        "disponibles": df.notna().sum(),
        "faltantes": df.isna().sum(),
        "porcentaje_faltante": df.isna().mean() * 100,
    }
)

missing_report = missing_report.sort_values(
    "porcentaje_faltante",
    ascending = False,
)

print("Número de filas analizadas: ", len(df))
print("Columnas con al menos un faltante: ", (missing_report["faltantes"] > 0).sum())
display(
    missing_report.loc[
        missing_report["faltantes"] > 0
    ].round(2)
)

Número de filas analizadas:  11143
Columnas con al menos un faltante:  6


,disponibles,faltantes,porcentaje_faltante
b_sigma,0,11143,100.00
M_sigma,9501,1642,14.74
T_sigma,9501,1642,14.74
T (years),10363,780,7.00
M (deg),10363,780,7.00
node_sigma,11141,2,0.02


In [ ]:
# Revisar la disponiblidad de las variables importantes para el análisis
analysis_columns = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (code)",
    "Sol lon (deg)",
    "RAgeo (deg)",
    "RAgeo_sigma",
    "DECgeo (deg)",
    "DECgeo_sigma",
    "LAMgeo (deg)",
    "BETgeo (deg)",
    "Vgeo (km/s)",
    "Vgeo_sigma",
    "Peak (AbsMag)",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
]

analysis_missing_report = missing_report.loc[analysis_columns].copy()

display(analysis_missing_report.round(2))

,disponibles,faltantes,porcentaje_faltante
Unique trajectory (identifier),11143,0,0.0
Beginning (UTC Time),11143,0,0.0
IAU (code),11143,0,0.0
Sol lon (deg),11143,0,0.0
RAgeo (deg),11143,0,0.0
RAgeo_sigma,11143,0,0.0
DECgeo (deg),11143,0,0.0
DECgeo_sigma,11143,0,0.0
LAMgeo (deg),11143,0,0.0
BETgeo (deg),11143,0,0.0


In [50]:
# Detectar filas duplicadas e identificadores de trayectoria repetidos sin eliminar observaciones

id_column = "Unique trajectory (identifier)"

duplicate_id_mask = df.duplicated(subset = [id_column], keep = False)

duplicate_summary = pd.DataFrame(
    {
        "medida": [
            "Filas totales",
            "Identificadores únicos",
            "Filas con un ID repetido",
            "Identificadores que aparecen más de una vez",
            "Filas adicionales por repetición de ID",
            "Filas completamente duplicadas adicionales"
        ],
        "valor": [
            len(df),
            df[id_column].nunique(dropna = False),
            duplicate_id_mask.sum(),
            df.loc[duplicate_id_mask, id_column].nunique(),
            df.duplicated(subset = [id_column], keep = "first").sum(),
            df.duplicated(keep = "first").sum(),
        ],
    }
)

display(duplicate_summary)

if duplicate_id_mask.any():
    duplicate_review_columns = [
        id_column,
        "Beginning (UTC Time)",
        "IAU (No)",
        "IAU (code)",
        "Sol lon (deg)",
        "RAgeo (deg)",
        "DECgeo (deg)",
        "Vgeo (km/s)",
        "Peak (AbsMag)",
        "source_file",
    ]
    
    duplicate_review = (
        df.loc[
            duplicate_id_mask,
            duplicate_review_columns,
        ]
        .sort_values(id_column)
    )
    
    print("Primeros identificadores repetidos: ")
    display(duplicate_review.head(20))
    
else:
    print("No se encontraron identificadores repetidos.")

,medida,valor
0,Filas totales,11143
1,Identificadores únicos,11143
2,Filas con un ID repetido,0
3,Identificadores que aparecen más de una vez,0
4,Filas adicionales por repetición de ID,0
5,Filas completamente duplicadas adicionales,0


No se encontraron identificadores repetidos.


In [53]:
# Comprobar que las variables principales estén dentro de rangos matemáticos y físicos válidos

reglas_validez = {
    "Longitud solar fuera de [0, 360)": (
        df["Sol lon (deg)"].ge(0)
        & df["Sol lon (deg)"].lt(360)
    ),
    
    "Ascensión recta fuera de [0, 360)": (
      df["RAgeo (deg)"].ge(0)
      & df["RAgeo (deg)"].lt(360)
    ),
    
    "Declinación fuera se [-90, 90]": (
        df["DECgeo (deg)"].between(-90,90)
    ),
    
    "Longitud eclíptica fuera de [0, 360)": (
        df["LAMgeo (deg)"].ge(0)
        & df["LAMgeo (deg)"].lt(360)
    ),
    
    "Latitud eclíptica fuera de [-90, 90]": (
        df["BETgeo (deg)"].between(-90, 90)
    ),
    
    "Velocidad geocéntrica no positiva": (
        df["Vgeo (km/s)"].gt(0)
    ),
    
    "Incertidumbre de RA negativa": (
        df["RAgeo_sigma"].ge(0)
    ),
    
    "Incertidumbre de DEC negativa": (
        df["DECgeo_sigma"].ge(0)
    ),
    
    "Incertidumbre de velocidad negativa": (
        df["Vgeo_sigma"].ge(0)
    ),
    
    "Ángulo de convergencia fuera de [0, 180]": (
        df["Qc (deg)"].between(0, 180)
    ),
    
    "Error de ajuste negativo": (
        df["MedianFitErr (arcsec)"].ge(0)
    ),
    
    "Menos de dos estaciones": (
        df["Num (stat)"].ge(2)
    )
}

resultados_rangos = []

for descripcion, mascara_valida in reglas_validez.items():
    mascara_invalida = ~mascara_valida.fillna(False)
    cantidad_invalida = mascara_invalida.sum()
    
    resultados_rangos.append(
        {
            "comprobación": descripcion,
            "filas_invalidas": cantidad_invalida,
            "porcentaje_invalido": round(
                100 * cantidad_invalida / len(df),
                4,
            ),
        }
    )
    
auditoria_rangos = pd.DataFrame(resultados_rangos)

display(auditoria_rangos)

,comprobación,filas_invalidas,porcentaje_invalido
0,"Longitud solar fuera de [0, 360)",0,0.0
1,"Ascensión recta fuera de [0, 360)",0,0.0
2,"Declinación fuera se [-90, 90]",0,0.0
3,"Longitud eclíptica fuera de [0, 360)",0,0.0
4,"Latitud eclíptica fuera de [-90, 90]",0,0.0
5,Velocidad geocéntrica no positiva,0,0.0
6,Incertidumbre de RA negativa,0,0.0
7,Incertidumbre de DEC negativa,0,0.0
8,Incertidumbre de velocidad negativa,0,0.0
9,"Ángulo de convergencia fuera de [0, 180]",0,0.0
